# scRNA-seq Teaching Pipeline -- Interactive Walkthrough

This notebook follows the same order as the course slide deck (`SC_analysis.pptx`), one section per concept, running real code on real data instead of just showing snippets.

**Before running this notebook**, make sure you have:
1. Set up WSL and the `scrna-teaching` conda environment (`setup/install_env.sh` -- see `GUIDE.md`).
2. Selected the **Python (scrna-teaching)** kernel for this notebook (Kernel -> Change Kernel).

By default this notebook uses **pbmc3k**, a real 10x Genomics droplet dataset (2,700 PBMCs, raw UMI counts), downloaded automatically by Scanpy from 10x Genomics' own servers -- so every concept in the slides (mitochondrial QC, doublets, batch correction, annotation) has real data to run on.

Section 9 at the end shows how to instead point the notebook at **the sample dataset from the course GitHub repository** (`GiatrasKon/scRNAseq-Analysis-Pipeline`), and explains why a few raw-count-only steps don't apply to it.

In [ ]:
import sys
sys.path.append("../scripts")

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

from utils import load_config, is_outlier, looks_like_raw_counts, has_gene_symbol_annotation

cfg = load_config("../config.yaml")
sc.settings.verbosity = 1
sc.settings.figdir = "../results/figures"
sc.set_figure_params(dpi=90, facecolor="white")
cfg["dataset"]

## 1) What is Single-Cell RNA Sequencing?

Bulk RNA-seq averages expression across millions of cells and loses cell-to-cell variation. scRNA-seq measures each cell separately, so we can identify **distinct cell types, rare populations, tumor heterogeneity, immune infiltration, and developmental trajectories**.

## 2) From FASTQ to Gene-Cell Matrix

Raw sequencing reads (FASTQ) are aligned and counted by tools like **Cell Ranger**, **STARsolo**, or **kallisto\|bustools**, which extract the cell barcode and UMI from each read, remove PCR duplicates using the UMI, and produce a **gene x cell count matrix**. This notebook picks up *after* that step -- `sc.datasets.pbmc3k()` downloads Cell-Ranger-processed 10x output directly.

In [ ]:
sc.settings.datasetdir = "../data/raw"
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()
print(adata)

## 3) & 4) File Formats and the AnnData Object

```
adata
 |-- X          expression matrix (cells x genes)
 |-- obs        cell metadata
 |-- var        gene metadata
 |-- obsm        multi-dimensional embeddings (PCA, UMAP)
 |-- uns         unstructured info (clustering params, etc.)
 |-- layers      different versions of the expression matrix
 `-- raw         a frozen copy of the original data
```

## 5) Dense vs Sparse Matrix

Single-cell data is mostly zeros -- most genes are not expressed in most cells. A **sparse matrix** stores only non-zero values (memory efficient); a **dense matrix** stores everything. `adata.X` starts sparse; convert only a small slice to dense when you actually need to (e.g. for a plotting library that requires a dense array):

In [ ]:
X_counts_sample = adata.X[:5, :5]
if not isinstance(X_counts_sample, np.ndarray):
    X_counts_sample = X_counts_sample.toarray()
X_counts_sample

## 6) Quality Control

Single-cell experiments contain dead/broken cells, empty droplets, doublets, and ambient RNA contamination. Left in, they create fake clusters, false DE genes, and wrong biological conclusions.

**Step 1: flag mitochondrial / ribosomal / hemoglobin genes.** Dying cells leak cytoplasmic RNA while mitochondrial RNA remains, so a high mitochondrial percentage flags low-quality cells.

In [ ]:
adata.var["mt"] = adata.var_names.str.upper().str.startswith(("MT-", "MT."))
adata.var["ribo"] = adata.var_names.str.upper().str.startswith(("RPS", "RPL"))
adata.var["hb"] = adata.var_names.str.upper().str.contains(r"^HB[^(P)]")
print(f"mitochondrial genes: {adata.var['mt'].sum()}, ribosomal: {adata.var['ribo'].sum()}, hemoglobin: {adata.var['hb'].sum()}")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], percent_top=[20], log1p=True, inplace=True)
adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe()

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True)

**Step 2: MAD-based outlier detection.** The Median Absolute Deviation is robust to extreme values (unlike standard deviation). We flag cells more than `nmads` MADs from the median -- removing extremely low cells (empty droplets) and extremely high cells (possible doublets).

In [ ]:
nmads = cfg["qc"]["mad_nmads"]
adata.obs["outlier"] = (
    is_outlier(adata.obs["log1p_total_counts"], nmads)
    | is_outlier(adata.obs["log1p_n_genes_by_counts"], nmads)
    | is_outlier(adata.obs["pct_counts_in_top_20_genes"], nmads)
)
adata.obs["mt_outlier"] = is_outlier(adata.obs["pct_counts_mt"], nmads) | (adata.obs["pct_counts_mt"] > cfg["qc"]["max_pct_mt"])
print(f"statistical outliers: {adata.obs['outlier'].sum()}, high-mito outliers: {adata.obs['mt_outlier'].sum()} (of {adata.n_obs} cells)")

n_before = adata.n_obs
adata = adata[~(adata.obs["outlier"] | adata.obs["mt_outlier"])].copy()
sc.pp.filter_cells(adata, min_genes=cfg["qc"]["min_genes_per_cell"])
sc.pp.filter_genes(adata, min_cells=cfg["qc"]["min_cells_per_gene"])
print(f"cells kept: {n_before} -> {adata.n_obs}")

## 7) Doublets

A doublet happens when two cells enter the same droplet, producing an artificial hybrid transcriptome that can create fake clusters. **Scrublet** simulates artificial doublets by combining pairs of real cells, then scores every real cell by how similar it looks to a simulated doublet.

In [ ]:
sc.pp.scrublet(adata, expected_doublet_rate=cfg["doublets"]["expected_doublet_rate"])
print(f"predicted doublets: {int(adata.obs['predicted_doublet'].sum())} / {adata.n_obs}")
adata = adata[~adata.obs["predicted_doublet"]].copy()

## 8) Normalization, Log Transform, Highly Variable Genes

Each cell has a different sequencing depth (Cell A: 50,000 reads, Cell B: 10,000 reads) -- comparing raw counts would just measure that technical difference, not biology. `normalize_total` scales every cell to the same total count; `log1p` compresses the long tail of highly expressed genes so a few extreme values don't dominate PCA; and highly-variable-gene selection keeps the ~2000 genes that actually vary across cells instead of flat housekeeping genes.

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=cfg["normalize"]["target_sum"])
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()

sc.pp.highly_variable_genes(adata, n_top_genes=cfg["hvg"]["n_top_genes"], subset=True, flavor=cfg["hvg"]["flavor"])
print(f"kept {adata.n_vars} highly variable genes")

## 9) Scaling and PCA

Scaling subtracts the mean and divides by the standard deviation per gene (mean 0, variance 1) so PCA -- which is sensitive to magnitude -- doesn't let large-value genes dominate. PCA then finds the directions (linear combinations of genes) that explain the most variance, reducing ~2000 gene dimensions down to 30-50 principal components: removing noise, capturing major biological variation, and making everything downstream faster.

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=cfg["dimred"]["n_pcs"], svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, n_pcs=cfg["dimred"]["n_pcs"], log=True)

## 10) Neighbors and UMAP

Using the top PCs, we find each cell's k nearest neighbors and build a graph -- clustering and UMAP both operate on this graph. UMAP (Uniform Manifold Approximation and Projection) then lays that graph out in 2D, preserving local neighborhoods (and some global structure) so we can visually spot cell populations, batch effects, and heterogeneity.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=cfg["dimred"]["n_neighbors"], n_pcs=cfg["dimred"]["n_pcs"])
sc.tl.umap(adata)
sc.pl.umap(adata, color=["pct_counts_mt", "total_counts"])

## 11) Clustering (Leiden) and Differential Expression

Leiden starts with every cell as its own cluster, then merges cells to maximize **modularity** (are connections stronger within a cluster than between clusters?). **Resolution** controls granularity: low resolution -> few large clusters, high resolution -> many small clusters. `rank_genes_groups` then runs a Wilcoxon rank-sum test, comparing each cluster's cells against all other cells to find marker genes.

In [ ]:
sc.tl.leiden(adata, resolution=cfg["clustering"]["resolution"], flavor="igraph", n_iterations=2)
sc.pl.umap(adata, color="leiden", legend_loc="on data")

sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

## 12) Batch Effects

A batch effect is technical variation unrelated to biology (different reagents, day, sequencing depth) that can make a UMAP separate by batch instead of by cell type. **BBKNN** forces each cell's neighbors to be drawn evenly from every batch; **Harmony** instead adjusts the PCA embedding directly, shifting each batch to align while preserving biological structure.

pbmc3k has no real batch variable, so the cell below creates a **synthetic** 2-way split purely to demonstrate the mechanics -- do not read biological meaning into the result.

In [ ]:
rng = np.random.default_rng(42)
adata.obs["batch"] = rng.choice(["batch_1", "batch_2"], size=adata.n_obs)

sc.external.pp.harmony_integrate(adata, key="batch")
sc.pp.neighbors(adata, use_rep="X_pca_harmony")
sc.tl.umap(adata)
sc.pl.umap(adata, color=["batch", "leiden"])

## 13) Cell Type Annotation

After clustering, clusters are just numbers (0, 1, 2, ...) -- annotation assigns biological meaning.

**Manual annotation** scores canonical marker genes per cluster (e.g. high `CD3E` -> T cells, high `CD19` -> B cells).

In [ ]:
marker_genes = {
    "T cells": ["CD3D", "CD3E", "IL7R"],
    "CD14+ Monocytes": ["CD14", "LYZ"],
    "B cells": ["CD19", "MS4A1", "CD79A"],
    "NK cells": ["GNLY", "NKG7"],
    "FCGR3A+ Monocytes": ["FCGR3A", "MS4A7"],
    "Dendritic cells": ["FCER1A", "CST3"],
    "Megakaryocytes": ["PPBP"],
}
available = {k: [g for g in v if g in adata.var_names] for k, v in marker_genes.items()}
available = {k: v for k, v in available.items() if v}
for cell_type, genes in available.items():
    sc.tl.score_genes(adata, gene_list=genes, score_name=f"score_{cell_type.replace(' ', '_')}")

score_cols = [f"score_{k.replace(' ', '_')}" for k in available]
cluster_scores = adata.obs.groupby("leiden", observed=True)[score_cols].mean()
best_type = cluster_scores.idxmax(axis=1).str.replace("score_", "").str.replace("_", " ")
adata.obs["manual_annotation"] = adata.obs["leiden"].map(best_type.to_dict()).astype("category")
sc.pl.umap(adata, color="manual_annotation")

**Machine-learning based annotation (CellTypist)**: a logistic-regression classifier pretrained on large labeled reference atlases. Steps: normalize input data, match genes to the model's genes, compute a probability per cell type, assign the highest-probability label. `predicted_labels` gives a cell-level prediction; `majority_voting` refines that to the most frequent label within each cluster, reducing noise.

The slide's own example loads `celltypist.models.Model.load("Cells_Adult_Breast.pkl")` for a *breast* dataset. pbmc3k is PBMC/immune data, so we use `Immune_All_Low.pkl` instead -- always pick the model that matches your tissue (full list: https://www.celltypist.org/models). This cell is optional and downloads a model (~tens of MB) on first use.

In [ ]:
RUN_CELLTYPIST = cfg["annotation"]["run_celltypist"]  # flip to True (or edit config.yaml) to run this cell

if RUN_CELLTYPIST:
    import celltypist
    from celltypist import models

    model_name = cfg["annotation"]["celltypist_model"]
    models.download_models(model=model_name)
    model = models.Model.load(model_name)

    adata_ct = adata.copy()
    adata_ct.X = adata_ct.layers["lognorm"]
    predictions = celltypist.annotate(adata_ct, model=model, majority_voting=True)
    result = predictions.to_adata()
    adata.obs["celltypist_majority_voting"] = result.obs["majority_voting"]
    sc.pl.umap(adata, color="celltypist_majority_voting")
else:
    print("Skipped -- set annotation.run_celltypist: true in config.yaml (or RUN_CELLTYPIST = True above) to run it.")

In [ ]:
import pathlib
out_path = pathlib.Path("../data/processed/adata_notebook_final.h5ad")
out_path.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(out_path)
print(f"Saved -> {out_path}")

## 14) Using the GitHub sample dataset instead

The course GitHub repository (https://github.com/GiatrasKon/scRNAseq-Analysis-Pipeline) ships a 137-cell x 54,675-probe dataset (`data/RNA-seq.csv.gz`) built from Affymetrix microarray probes that are **already log2-normalized** -- not raw 10x UMI counts, and not annotated with human gene symbols.

That means:
- Mitochondrial/ribosomal/hemoglobin QC (Section 6) does not apply -- there are no `MT-`/`RPS`/`RPL`/`HB`-prefixed gene symbols to match against, only probe IDs like `1007_s_at`.
- Scrublet doublet detection (Section 7) does not apply -- it needs raw integer counts to simulate doublets by summation; this data is already continuous, normalized log-intensities.
- `normalize_total`/`log1p` (Section 8) should be **skipped** -- the data is already normalized, so re-normalizing would distort it.
- Everything else (HVG, scaling, PCA, neighbors, UMAP, Leiden, DE, manual annotation) still works the same way.

To run the full pipeline (not just this notebook) on this dataset instead, set `dataset: "github_sample"` in `config.yaml` and re-run `bash run_pipeline.sh` -- every stage script already detects these differences automatically (see `scripts/utils.py`) and skips the inapplicable steps with an explanation, rather than silently producing meaningless numbers.

This dataset also ships a `data/labels.csv` ground-truth clustering from the original assignment, and the pipeline's clustering stage (`scripts/08_clustering_de.py`) automatically compares Leiden's clusters against it (Adjusted Rand Index / Normalized Mutual Information), and reproduces the original repo's own GMM/DBSCAN-on-PCA approach for a side-by-side comparison -- a good way to see how the graph-based Leiden workflow taught in this course stacks up against the classical ML clustering used in the original assignment.